In [0]:
import boto3
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""

AWS_BUCKET_NAME = ""

s3 = boto3.client(
    "s3",
    aws_access_key_id="",
    aws_secret_access_key=""
)

In [0]:
from pyspark.sql.functions import *
bronze_df = spark.table("bronze_youtube_events")

In [0]:
bronze_df.show(5, truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|ingestion_time            |
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+
|India  |Desktop    |2026-05-28 09:20:53.245290|false|false     |2368   |Education     |vid_3   |Hindi         |745       |2026-05-28 09:24:19.259333|
|India  |Tablet     |2026-05-28 09:20:53.245322|false|true      |7696   |Gaming        |vid_35  |Spanish       |781       |2026-05-28 09:24:19.259333|
|India  |Desktop    |2026-05-28 09:20:53.245334|true |true      |3587   |Education     |vid_45  |English       |806       |2026-05-28 09:24:19.259333|
|India  |Mobile     |2026-05-28 09:20:53.245343|true |true      |1793   |Vlogs         |vid_13

In [0]:
bronze_df.count()

1000

In [0]:
silver_df = bronze_df.filter(
    col("watch_time") > 0
)

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.withColumn(
    "event_time",
    to_timestamp("event_time")
)

In [0]:
silver_df = silver_df.withColumn(
    "engagement_score",
    when(col("liked") == True, 1).otherwise(0) +
    when(col("subscribed") == True, 2).otherwise(0)
)

In [0]:
silver_df.show(5, truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+----------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|ingestion_time            |engagement_score|
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+----------------+
|India  |Desktop    |2026-05-28 09:20:53.24529 |false|false     |2368   |Education     |vid_3   |Hindi         |745       |2026-05-28 09:24:19.259333|0               |
|India  |Tablet     |2026-05-28 09:20:53.245322|false|true      |7696   |Gaming        |vid_35  |Spanish       |781       |2026-05-28 09:24:19.259333|2               |
|India  |Desktop    |2026-05-28 09:20:53.245334|true |true      |3587   |Education     |vid_45  |English       |806       |2026-05-28 09:24:19.259333|3         

In [0]:
silver_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- liked: boolean (nullable = true)
 |-- subscribed: boolean (nullable = true)
 |-- user_id: long (nullable = true)
 |-- video_category: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_language: string (nullable = true)
 |-- watch_time: long (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- engagement_score: integer (nullable = false)



In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_youtube_events")

In [0]:
spark.sql("""
SELECT *
FROM silver_youtube_events
LIMIT 5
""").show(truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+----------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|ingestion_time            |engagement_score|
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+----------------+
|USA    |Desktop    |2026-05-28 09:20:53.245397|true |false     |4637   |Tech          |vid_3   |Hindi         |302       |2026-05-28 09:24:19.259333|1               |
|Germany|Desktop    |2026-05-28 09:20:53.24556 |false|false     |5534   |Vlogs         |vid_21  |Hindi         |798       |2026-05-28 09:24:19.259333|0               |
|UK     |Mobile     |2026-05-28 09:20:53.245576|true |true      |5413   |Gaming        |vid_30  |Spanish       |1109      |2026-05-28 09:24:19.259333|3         

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM silver_youtube_events
""").show()

+--------+
|COUNT(*)|
+--------+
|    1000|
+--------+



In [0]:
silver_export = silver_df.toPandas()

silver_export.to_csv(
    "/tmp/silver_youtube_events.csv",
    index=False
)

s3.upload_file(
    "/tmp/silver_youtube_events.csv",
    "youtube-analytics-lakehouse",
    "silver/silver_youtube_events.csv"
)

print("Silver exported to S3")

Silver exported to S3
